In [1]:
import pandas as pd
from scipy.io import loadmat
import pandas as pd
import re
import networkx as nx
import numpy as np
from collections import OrderedDict
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import concatenate_datasets, load_dataset
from datasets import Dataset, DatasetDict
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig, Trainer, TrainingArguments
from datasets import Dataset
from scipy.stats import pearsonr, spearmanr
import ast
import sys
from utils.prompting import *
from reward.helpful_opinion import *

2026-01-13 19:21:52.418434: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 19:21:52.444417: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 19:21:52.444448: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 19:21:52.444464: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 19:21:52.450519: I tensorflow/core/platform/cpu_feature_g

In [2]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


# Read Data

In [3]:
model_checkpoint = "microsoft/deberta-v2-xlarge"   # swap to "bert-base-uncased" if you prefer BERT
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
ground_truth_df = pd.read_pickle("../data/test/test.pkl")
ground_truth_df = ground_truth_df[['category', 'product_name', 'user_id', 'claim_split_gold']]
ground_truth_df = ground_truth_df.rename(columns={'claim_split_gold': 'key_point_given'})
ground_truth_df

,category,product_name,user_id,key_point_given
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[Head & Shoulders Normal Hair Shampoo contains...
1,Beauty,Gillette Mach 3 Razor,5000858,[Swapping blades is easy with the single-point...
2,Beauty,Pitrok,5296801,[PitRok is gentle and effective for sensitive ...
3,Beauty,Gillette Mach 3 Razor,5050855,[Replacement blades are more expensive compare...
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[The shampoo gently enhances natural highlight...
...,...,...,...,...
95,Travel,Dollar Rent A Car Worldwide,5297771,[Dollar Rent A Car Worldwide offers consistent...
96,Travel,Amsterdam (Netherlands),5202501,[Accommodation options range from budget-frien...
97,Travel,Leicester in General,5020891,[Leicester offers a surprisingly rich mix of e...
98,Travel,Milan in general,5091015,[Public transport in Milan is generally excell...


In [5]:
root_path = f"../output/stage_2_rl_inference/summary_kp_extraction"
temp_df = pd.read_pickle(root_path + "/1/1_done.pkl")
temp_df = temp_df.rename(columns={'voter_full': 'user_id'})
temp_df.shape

(96, 9)

In [6]:
temp_df['claim_split_predicted'] = temp_df['claim_split_predicted'].apply(lambda x: x.strip("```json").strip("\n"))
temp_df['claim_split_predicted'] = temp_df['claim_split_predicted'].apply(lambda x: ast.literal_eval(x))

In [7]:
temp_df = temp_df.rename(columns={'claim_split_predicted': 'key_point'})

In [8]:
claim_split_predicted = temp_df.merge(ground_truth_df)

In [9]:
claim_split_predicted

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,[Head & Shoulders Normal Hair Shampoo is effec...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...
1,[The Gillette Mach 3 Razor provides a close an...,Based on the user profile and the helpful key ...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1,[Swapping blades is easy with the single-point...
2,[Pitrok is a natural deodorant free from artif...,Based on the helpful key points and the user p...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1,[PitRok is gentle and effective for sensitive ...
3,[The Gillette Mach 3 Razor delivers a close an...,Here is a personalized summary of product A (G...,Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1,[Replacement blades are more expensive compare...
4,"[The shampoo is gentle and effective., It is a...",Based on the user profile and the helpful key ...,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1,[The shampoo gently enhances natural highlight...
...,...,...,...,...,...,...,...,...,...,...
91,[Dollar Rent A Car Worldwide is a reliable and...,Based on the helpful key points and user 111's...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1,[Dollar Rent A Car Worldwide offers consistent...
92,[Amsterdam is steeped in tradition and is very...,Here is a personalized summary of product A (A...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1,[Accommodation options range from budget-frien...
93,[Leicester offers a diverse range of shopping ...,Here is a personalized summary of product A (L...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1,[Leicester offers a surprisingly rich mix of e...
94,[Milan features stunning architecture that imp...,Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,[Public transport in Milan is generally excell...


In [10]:
mask = claim_split_predicted['filtered_hist_vote_written'].str.len() == 0
display(claim_split_predicted[mask].shape)
claim_split_predicted = claim_split_predicted[~mask]

(12, 10)

In [11]:
merged_df = claim_split_predicted.explode(['key_point'])

In [13]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_rouge_score(row):
    rouge1_scores, rouge2_scores, rougel_scores = [], [], []
    for rev in row['filtered_hist_vote_written']:
        scores = scorer.score(row['key_point'], rev)
        rouge1_scores += [scores['rouge1'].fmeasure]
        rouge2_scores += [scores['rouge2'].fmeasure]
        rougel_scores += [scores['rougeL'].fmeasure]
        
    row['rouge1_scores'] = rouge1_scores
    row['rouge2_scores'] = rouge2_scores
    row['rougel_scores'] = rougel_scores

    return row

In [14]:
def join_history(reviews):
    # Use tokenizer-specific separator; helps the encoder segment reviews
    sep = tokenizer.sep_token if tokenizer.sep_token is not None else " "
    return f" {sep} ".join([r.strip() for r in reviews])

In [15]:
merged_df = merged_df.parallel_apply(calculate_rouge_score, axis=1)

In [16]:
merged_df

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given,rouge1_scores,rouge2_scores,rougel_scores
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...,"[0.06802721088435373, 0.037267080745341616, 0....","[0.0, 0.0, 0.010152284263959392, 0.04145077720...","[0.06802721088435373, 0.024844720496894408, 0...."
0,"The shampoo leaves hair soft, clean, and shiny.",Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...,"[0.055944055944055944, 0.025477707006369428, 0...","[0.0, 0.0, 0.010362694300518135, 0.01058201058...","[0.04195804195804196, 0.025477707006369428, 0...."
0,It works well for dry and curly hair.,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...,"[0.055944055944055944, 0.050955414012738856, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00904977...","[0.04195804195804196, 0.03821656050955414, 0.0..."
0,The shampoo is easy to use.,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...,"[0.07092198581560283, 0.03870967741935484, 0.0...","[0.014388489208633093, 0.0, 0.0418848167539267...","[0.07092198581560283, 0.03870967741935484, 0.0..."
0,"It is available in different styles, including...",Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...,"[0.14379084967320263, 0.0718562874251497, 0.10...","[0.013245033112582783, 0.0, 0.0197044334975369...","[0.06535947712418301, 0.04790419161676647, 0.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,"Milan offers delicious food, including a wide ...",Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,[Public transport in Milan is generally excell...,"[0.07361963190184048, 0.04745762711864407, 0.0...","[0.012422360248447206, 0.0, 0.0, 0.0]","[0.06134969325153374, 0.03389830508474576, 0.0..."
94,Milan provides an authentic Italian dining exp...,Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,[Public transport in Milan is generally excell...,"[0.03870967741935484, 0.013937282229965157, 0....","[0.0, 0.0, 0.0, 0.0]","[0.02580645161290323, 0.013937282229965157, 0...."
94,The city has a strong sense of community that ...,Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,[Public transport in Milan is generally excell...,"[0.08588957055214724, 0

In [17]:
merged_df_full = merged_df[::]

In [18]:
def filter_hist_vote_written(row):
    my_df = row[['filtered_hist_vote_written', 'rouge1_scores', 'rouge2_scores', 'rougel_scores']].to_frame().T
    my_df = my_df.explode(['filtered_hist_vote_written', 'rouge1_scores', 'rouge2_scores', 'rougel_scores'])
    thres = 0.06
    mask = my_df['rougel_scores'] >= thres
    mask &= my_df['rouge2_scores'] >= thres
    mask &= my_df['rouge1_scores'] >= thres
    my_df = my_df[mask]
    
    my_df = my_df.sort_values(by=['rougel_scores', 'rouge2_scores', 'rouge1_scores'], ascending=False)
    row['sorted_hist_vote_written'] = my_df['filtered_hist_vote_written'].tolist()
    row['sorted_rougel_scores'] = my_df['rougel_scores'].tolist()
    return row

In [19]:
merged_df = merged_df_full.parallel_apply(filter_hist_vote_written, axis=1)

In [20]:
merged_df = merged_df[merged_df['sorted_hist_vote_written'].str.len() > 0]

In [22]:
merged_df.shape

(13, 15)

In [24]:
merged_df = merged_df[['category', 'product_name', 'user_id', 'key_point', 'sorted_hist_vote_written']].\
    rename(columns={'sorted_hist_vote_written': 'hist_vote_written'})

# Calculate Opinion Helpfulness (DeBERTa)

## Setup

In [25]:
class CrossEncoderRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob if hasattr(self.config, "hidden_dropout_prob") else 0.1)
        self.head = nn.Linear(self.config.hidden_size, 1)  # scalar
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # pool: prefer CLS token; if pooler exists (BERT), you can use pooler_output
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = out.last_hidden_state[:, 0, :]  # CLS
        x = self.dropout(pooled)
        raw = self.head(x).squeeze(-1)             # shape [B]
        pred01 = torch.sigmoid(raw)                 # in [0,1]
        pred = pred01 * 4.0 + 1.0                  # bound to [1,5]
#         pred = pred01 * 5.0    # ----> now in [0,5]
#         pred = torch.sigmoid(raw) * 5.0    # ----> now in [0,5]
        outputs = {"logits": pred.unsqueeze(-1)}   # Trainer expects "logits"
        if labels is not None:
            labels = labels.to(pred.dtype)
            loss = F.mse_loss(pred, labels)
            outputs["loss"] = loss
        return outputs

In [26]:
model = CrossEncoderRegressor("microsoft/deberta-v2-xlarge")
model.load_state_dict(torch.load('../models/stage_2_helpful_opinion_reward_deberta_ft/model.pth'))

/tmp/ipykernel_8103/3086173370.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../models/stage_2_helpful_opinion_reward_deberta_ft/mode

<All keys matched successfully>

In [27]:
import torch
if torch.cuda.is_available():       
    device = torch.device("cuda")
    print("Using GPU.")
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")
model.to(device)

Using GPU.


CrossEncoderRegressor(
  (backbone): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 1536, padding_idx=0)
      (LayerNorm): LayerNorm((1536,), eps=1e-07, elementwise_affine=True)
      (dropout): StableDropout()
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-23): 24 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=1536, out_features=1536, bias=True)
              (key_proj): Linear(in_features=1536, out_features=1536, bias=True)
              (value_proj): Linear(in_features=1536, out_features=1536, bias=True)
              (pos_dropout): StableDropout()
              (dropout): StableDropout()
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=1536, out_features=1536, bias=True)
              (LayerNorm): LayerNorm((1536,), eps=1e-07, elementwise_affine=

In [28]:
merged_df['history_text'] = merged_df['hist_vote_written'].apply(join_history)
merged_df = merged_df.rename(columns={'key_point': 'kp', 'voter_full': 'user_id'})
eval_data = Dataset.from_pandas(merged_df)

In [29]:
tokenizer.model_max_length

1000000000000000019884624838656

In [30]:
MAX_LEN = 512

In [31]:
def tokenize_function(examples):
    out = tokenizer(
        examples["kp"],
        examples["history_text"],
        padding="max_length", 
        truncation=True,
        max_length=MAX_LEN,
    )
    out["user_id"] = examples["user_id"]
    return out

tokenized_eval_data = eval_data.map(tokenize_function, batched=True, remove_columns=eval_data.column_names)

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

In [32]:
tokenized_eval_data.set_format("torch")

In [33]:
from torch.utils.data import DataLoader

## Inference

In [34]:
eval_dataset = tokenized_eval_data.shuffle(seed=42)
eval_dataloader = DataLoader(eval_dataset, batch_size=2)

In [35]:
model.eval()
output = []
for batch in eval_dataloader:
    batch_inputs = batch['input_ids'].to(device)
    batch_masks = batch['attention_mask'].to(device)
    with torch.no_grad():
        output += model(batch_inputs, batch_masks)["logits"].view(1,-1).tolist()[0]

In [36]:
merged_df['predicted_helpfulness'] = output

In [37]:
merged_df

,category,product_name,user_id,kp,hist_vote_written,history_text,predicted_helpfulness
1,Beauty,Gillette Mach 3 Razor,5000858,The Mach 3 Razor is effective at providing a c...,[ The shave you get with a Gillette Mach 3 ...,The shave you get with a Gillette Mach 3 Razor...,3.750257
6,Books,Life Expectancy - Dean Koontz,6359157,The book is a thrilling page-turner that keeps...,[ This book has a great storeyline ( I do n...,This book has a great storeyline ( I do not kn...,3.662663
22,DVDs,Hannibal (DVD),10307,Fans of the Silence of the Lambs will apprecia...,[ I was eagerly awaiting the arrival of Han...,I was eagerly awaiting the arrival of Hannibal...,3.817113
23,DVDs,Saving Private Ryan (DVD),5001532,The film is a must-see for anyone interested i...,[ This film is brilliant. The special effec...,This film is brilliant. The special effects ar...,3.652746
25,Education & Careers,London University College,5003286,The cost of living in London is high.,[ I m in my second year of a 4 year degree ...,I m in my second year of a 4 year degree at UC...,3.646703
32,Electronics,Sony MiniDisc recorder,5217314,The recorder offers features for customizing t...,[ i brought the sony minidisc walkman at th...,i brought the sony minidisc walkman at the sta...,3.511563
33,Electronics,Apple iPod shuffle 1 GB,6417425,"The sound quality is excellent, with crisp and...","[ Wow. Touch meets iPod, a perfect combinat...","Wow. Touch meets iPod, a perfect combination ...",3.888734
50,Food & Drink,Cadbury Twirl,5719657,"The ingredients are simple and include milk, s...",[ Cadbury s Twirl is one of my top 5 chocol...,Cadbury s Twirl is one of my top 5 chocolate b...,3.582143
71,Household Appliances,Tefal Tefal,5023460,The Tefal Steamer allows you to cook a variety...,[ This multilayered steamer allows you to c...,This multilayered steamer allows you to cook m...,4.029932
72,Household Appliances,Bissell Carpet Cleaner,5020889,The Bissell Carpet Cleaner is easy to use and ...,[ I am a VERY big fan of all Bissell produc...,I am a VERY big fan of all Bissell products. I...,3.629767


In [38]:
helpfulness_df = merged_df.groupby(['category', 'product_name', 'user_id'])['predicted_helpfulness'].mean().reset_index()

# Calculate SHS

In [39]:
helpfulness_df

,category,product_name,user_id,predicted_helpfulness
0,Beauty,Gillette Mach 3 Razor,5000858,3.750257
1,Books,Life Expectancy - Dean Koontz,6359157,3.662663
2,DVDs,Hannibal (DVD),10307,3.817113
3,DVDs,Saving Private Ryan (DVD),5001532,3.652746
4,Education & Careers,London University College,5003286,3.646703
5,Electronics,Apple iPod shuffle 1 GB,6417425,3.888734
6,Electronics,Sony MiniDisc recorder,5217314,3.511563
7,Food & Drink,Cadbury Twirl,5719657,3.582143
8,Household Appliances,Bissell Carpet Cleaner,5020889,3.629767
9,Household Appliances,Tefal Tefal,5023460,4.029932


In [40]:
helpfulness_df.groupby(['category'])['predicted_helpfulness'].mean()

category
Beauty                  3.750257
Books                   3.662663
DVDs                    3.734930
Education & Careers     3.646703
Electronics             3.700148
Food & Drink            3.582143
Household Appliances    3.829850
Shopping                3.695319
Travel                  3.737112
Name: predicted_helpfulness, dtype: float64

In [42]:
helpfulness_df['predicted_helpfulness'].mean()

3.71700448791186